# End-to-End Exposure Calculation

This notebook demonstrates the core `geoexposure` workflow:

1. Constructing a synthetic GPS trajectory
2. Building a raster environment from land cover data
3. Fitting a mobility model to estimate spatial occupancy
4. Computing time-windowed exposure estimates
5. Visualising the results

All data here is synthetic, constructed directly in the notebook so that
the example is fully self-contained and reproducible without external files.


In [ ]:
# Imports
import datetime as dt

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import box

from geoexposure import KDE, Environment, GapMethod, LandCover, Scenario, SpatialData, Trajectory
from geoexposure.data.columns import DATETIME, X, Y

## 1. Trajectory

A `Trajectory` is a time-ordered sequence of GPS observations `(x, y, datetime)`
Here we construct a synthetic trajectory of 49 points moving diagonally across
a 1 km × 1 km grid over approximately four hours.

Dwell times are assigned using the **Voronoi** gap method, which gives each
point a dwell time equal to half the gap to the previous observation plus half
the gap to the next. This is the simplest strategy and requires no additional
parameters.

In [ ]:
# Create sample trajectory data
n = 49
start = dt.datetime(2020, 1, 1, 8, 0, 0)
times = [start + dt.timedelta(minutes=5 * i) for i in range(n)]
x = np.linspace(100.0, 900.0, n)
y = np.linspace(100.0, 900.0, n)
df = pd.DataFrame(
    {
        DATETIME: times,
        X       : x,
        Y       : y
    }
)
trajectory = Trajectory(df, source_id="example")
trajectory = trajectory.with_dwell_times(GapMethod.VORONOI)
print(trajectory.summary())

## 2. Environment

An `Environment` combines one or more `SpatialData` layers on a regular raster
grid. Each `SpatialData` layer wraps a vector GeoDataFrame and associates it
with one or more `Metric` objects that are evaluated on the grid.

Here we construct a simple 1 km × 1 km land cover dataset with three categories:

| Category | Geometry                                                    |
|----------|-------------------------------------------------------------|
| `forest` | Left half of the domain (x = 0–500, y=0-1000)               |
| `fields` | Top-right corner of the domain (x = 500–1000, y = 500-1000) |
| `water`  | Bottom-right corner of the domain (x = 500–1000, y = 0-500) |

A `LandCover` metric is attached to each category with equal weighting.
Setting `radius=0.0` means exposure is binary — a cell is either inside or
outside the target land cover, with no distance decay at the boundary.

`environment.calculate()` evaluates all metrics on the raster grid and caches.
the results to disk. This calculation wil be triggered automatically at a
later point in the exposure calculation if not manually called, but
pre-calculating can be helpful for finding bugs early.

In [ ]:
# 3 Build the environment

# Underlying spatial data
forest = box(0, 0, 500, 1000)
fields = box(500, 500, 1000, 1000)
water = box(500, 0, 1000, 500)
spatial_data_gdf = gpd.GeoDataFrame(
    {"land_type": ["forest", "fields", "water"]},
    geometry=[forest, fields, water],
    crs="EPSG:32648",
)

# Metrics related to specific land types
forest_metric = LandCover(radius=0.0, column="land_type", value="forest")
fields_metric = LandCover(radius=0.0, column="land_type", value="fields")
water_metric = LandCover(radius=0.0, column="land_type", value="water")
weighting = 1.0

# Construct environment and calculate metrics
spatial_data = {
    "land": SpatialData(
        spatial_data_gdf,
        metrics={
            forest_metric: weighting,
            fields_metric: weighting,
            water_metric: weighting,
        },
    )
}
environment = Environment(
    spatial_resolution=10,
    spatial_data=spatial_data,
    spatial_reference_data="land",
)
environment.calculate()

## 3. Mobility Model and Scenario

A **mobility model** estimates how a participant distributes their time across
the raster grid given their trajectory. Here we use **Kernel Density Estimation**
(KDE), which places a Gaussian kernel at each recorded position weighted by its
dwell time (computed with the voronoi method as described), then evaluates the
resulting density on the environment grid.

A `Scenario` bundles together a trajectory, environment, mobility model, gap
method, and timestep into a single object. Calling `scenario.run()` computes
the time-windowed exposure and returns a `ScenarioResult` containing:

- `result.exposure` — an `ExposureSeries` with one row per time window
- `result.occupancy_gdf()` — the normalised spatial density from the mobility model

In [ ]:
# 4 Define the mobility model and run the scenario
mobility = KDE(kernel="gaussian", bandwidth=50.0)
scenario = Scenario(
    trajectory=trajectory,
    environment=environment,
    mobility=mobility,
    gap_method=(GapMethod.VORONOI, None),
    timestep=dt.timedelta(minutes=10),
)
result = scenario.run()

## 4. Visualisation Spatial Data

The two panels below show:

- **Left** — the land cover environment with the trajectory overlaid as red points.
  The diagonal path crosses both the forest (top) and fields (bottom) regions.
- **Right** — the normalised occupancy distribution from the KDE mobility model.
  High density regions indicate cells where the participant is estimated to have
  spent more time.

In [ ]:
# 5 Visualise
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True)
fig.set_size_inches(12,  8)

# Environmental layer with trajectory points
ax = environment.plot_reference(ax=axes[0], column="land_type")
ax.scatter(trajectory.df["x"], trajectory.df["y"], color="red", marker='.')
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Environment with trajectory")

# Occupancy on raster grid derived with mobility model
occupancy = result.occupancy_gdf()
ax = occupancy.plot(ax=axes[1], column="density", cmap=plt.cm.Reds)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Occupancy from mobility model")

## 5. Exposure Time Series

The exposure series shows how much time (in seconds) the participant spent in
each land cover type within each 10-minute window. The two lines correspond to
the `forest` and `fields` metrics defined in the environment. The smooth
crossover at the centre arises from the 50m bandwidth in the KDE mobility
model.

Note that the values sum to approximately the window length (600 seconds) in
windows where the trajectory is fully within the domain, reflecting the
time-weighted nature of the exposure calculation.

In [ ]:
fig, ax = plt.subplots(1, 1)
fig.set_size_inches(12,  8)

df = result.exposure.dataframe
ax.plot(df["window_centre"], df[environment.columns], marker='o')
ax.legend(environment.columns, loc="center right")
ax.set_ylabel("Exposure in seconds per 10 minute window")

## 6. Aggregation

The `ExposureSeries` can be aggregated to any coarser time resolution by summing
raw exposure integrals across windows. Here we aggregate from 10-minute to
30-minute windows. The total accumulated exposure is preserved under aggregation.

In [ ]:
aggregated = result.exposure.aggregate(dt.timedelta(minutes=30))
aggregated.dataframe